# Enclosure good/defective detector: YOLO11n on Colab T4
Runtime -> Change runtime type -> **T4 GPU**. Upload `enclosure_dataset.zip` (from the repo, `training/`) when asked.
Dataset: my own photos, 70/20/10 group-aware split, boxes from `tools/label_enclosure.py`. Classes: 0 `enclosure_good`, 1 `enclosure_defective`.

In [ ]:
!nvidia-smi | head -12
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
from google.colab import files
up = files.upload()          # choose enclosure_dataset.zip
!rm -rf /content/enclosure_ds && mkdir /content/enclosure_ds && unzip -q enclosure_dataset.zip -d /content/enclosure_ds
!ls /content/enclosure_ds; !for s in train val test; do echo $s $(ls /content/enclosure_ds/$s/images | wc -l) images; done

In [ ]:
open('/content/enclosure_ds/data.yaml','w').write('''path: /content/enclosure_ds
train: train/images
val: val/images
test: test/images
names:
  0: enclosure_good
  1: enclosure_defective
''')
print(open('/content/enclosure_ds/data.yaml').read())

In [ ]:
# Settings (write them in the report): YOLO11n, imgsz 480, 100 epochs, patience 30, batch 16, seed 0.
# Try IMGSZ = 640 as a second run: small details (slots, pads) may need it, but it is slower on the Pi.
IMGSZ = 480
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data='/content/enclosure_ds/data.yaml', imgsz=IMGSZ, epochs=100, patience=30, batch=16, seed=0,
            project='/content/runs', name=f'enc_{IMGSZ}', exist_ok=True, plots=True)

In [ ]:
# Metrics on VAL (used to pick best.pt) and on the held-out TEST split (report this one)
best = YOLO(f'/content/runs/enc_{IMGSZ}/weights/best.pt')
for split in ('val', 'test'):
    m = best.val(data='/content/enclosure_ds/data.yaml', split=split, imgsz=IMGSZ, conf=0.25, plots=True, name=f'eval_{split}_{IMGSZ}', project='/content/runs', exist_ok=True)
    print(split, 'P %.3f R %.3f mAP50 %.3f mAP50-95 %.3f' % (m.box.mp, m.box.mr, m.box.map50, m.box.map))
    for k, ci in enumerate(m.box.ap_class_index):
        print('  ', m.names[int(ci)], 'AP50 %.3f' % m.box.ap50[k])

In [ ]:
# Wrong detections on the test split (for the '10 wrong detections' write-up)
import glob
res = best.predict(sorted(glob.glob('/content/enclosure_ds/test/images/*.jpg')), imgsz=IMGSZ, conf=0.25, save=True, project='/content/runs', name=f'test_pred_{IMGSZ}', exist_ok=True)
print(len(res), 'test images predicted; look in /content/runs/test_pred_%d' % IMGSZ)

In [ ]:
# Export ONNX (this is the M2 export) and time-stamp the file names
onnx_path = best.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print(onnx_path)

In [ ]:
!cd /content/runs && zip -q -r /content/enclosure_results.zip enc_480 eval_val_480 eval_test_480 test_pred_480
files.download('/content/enclosure_results.zip')
files.download(onnx_path)

After downloading: copy `best.onnx` to `app/` (as `enclosure.onnx`), unzip the results into `evidence/M1/06_train_and_run_detector/enclosure/`, write RESULTS.md (P, R, mAP50, confusion matrix) and WRONG_DETECTIONS.md (10 wrong detections with causes), then commit with `M1:` / `M2: export ONNX` in the message.